In [12]:
# Load packages
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict
import CCC_utils as CCC


In [13]:
# Paths

# CellPhoneDB files
ccc_dir_path = Path("C:/Users/gabri/Documents/Data_projects/MCcharactertization/ROSMAP/CellphoneDb_test/v5.0.0/cellphonedb")
#ccc_dir_path = Path("/Users/ga0028vi/Documents/MCcharactertization/ROSMAP/CellphoneDb_test/v5.0.0/cellphonedb")

# AGM gene-list folder
mainccc_dir = Path(f"C:/Users/gabri/Documents/Data_projects/MCcharactertization/ROSMAP/Top_sets/Set309/CellCellComms")
#mainccc_dir = Path(f"/Users/ga0028vi/Documents/MCcharactertization/ROSMAP/Top_sets/Set309/CellCellComms")

clusters_dir = Path(mainccc_dir, "CCC_genes")
cluster_path = clusters_dir / "PLS_significant_gene_lists_padded.csv"

interact_table = ccc_dir_path / "interaction_table.csv"
multi_table = ccc_dir_path / "multidata_table.csv"
protein_file = ccc_dir_path / "protein_table.csv"
gene_file = ccc_dir_path / "gene_table.csv"
complex_composition_file = ccc_dir_path / "complex_composition_table.csv"
complex_table_file = ccc_dir_path / "complex_table.csv"

out = Path(mainccc_dir, "CCC_results")
out.mkdir(parents=True, exist_ok=True)

In [14]:
#Load AGMs
cluster_df = pd.read_csv(cluster_path)

# make dict of cluster gene sets
cluster_gene_sets = {}
for col in cluster_df.columns:
    genes = cluster_df[col].dropna().astype(str)
    cluster_gene_sets[col] = set(genes)

# get AGM list
cluster_ids = list(cluster_df.columns)
print(f"Loaded {len(cluster_ids)} cell clusters")

Loaded 24 cell clusters


## Map protein and complex multidata IDs to genes

`interaction_table.csv` refers to both proteins and complexes using multidata IDs.

- `protein_table.csv` and `gene_table.csv` map a protein multidata ID to a gene.
- `complex_composition_table.csv` lists the protein components of each complex.
- `multidata_table.csv` provides the CellPhoneDB entity name and says whether the entity is a complex.
- `complex_table.csv` provides additional complex metadata, including stoichiometry.

In [15]:
# Protein Multidata ID -> Gene mapping
protein_df = pd.read_csv(protein_file)[["protein_name", "protein_multidata_id"]]

gene_df = pd.read_csv(gene_file)[["protein_name","gene_name"]]
gene_df = gene_df.drop_duplicates(subset=["protein_name", "gene_name"])

protein_gene_df = protein_df.merge(gene_df,on="protein_name",how="left")

# Every protein represented as a one-gene set.
protein_map = (protein_gene_df.groupby("protein_multidata_id")["gene_name"].apply(set).to_dict())

print(f"Mapped {len(protein_map)} protein multidata IDs")

Mapped 1355 protein multidata IDs


In [16]:
# Multidata entity metadata
# Read the master table containing both single proteins and complexes
multi_df = pd.read_csv(multi_table)

# map to bool if md2 is complex or a single gene protein
md2iscomplex = (multi_df.set_index("id_multidata")["is_complex"].to_dict())

print(f"Loaded metadata for {len(multi_df)} CellPhoneDB multidata entities")

Loaded metadata for 1717 CellPhoneDB multidata entities


In [17]:
# Complex multidata id -> all required component genes
# get df mapping protein ids to their complex ids
complex_composition_df = pd.read_csv(complex_composition_file)[["complex_multidata_id", "protein_multidata_id"]]

# Attach the component protein name and gene symbol to every complex row.
complex_component_df = complex_composition_df.merge(
    protein_gene_df[["protein_multidata_id", "protein_name", "gene_name"]],
    on="protein_multidata_id")

complex_map = (complex_component_df.groupby("complex_multidata_id")["gene_name"].apply(set).to_dict())

# Map each complex ID to a readable list of component proteins
complex_protein_components = (
    complex_component_df.groupby("complex_multidata_id")["protein_name"]
    .apply(lambda proteins: ";".join(sorted(set(proteins)))).to_dict())

# Combine individual proteins and complexes into one lookup
md2genes = {**protein_map, **complex_map}

print(f"Mapped {len(complex_map)} complexes")

Mapped 362 complexes


In [18]:
# Read interactions table
# Convert CellPhoneDB interaction partner IDs into gene sets.
records_by = defaultdict(list)
inter_df = pd.read_csv(interact_table)

for _, row in inter_df.iterrows():

    # Interaction category, e.g. LR or Adhesion
    cat = row["directionality"]
    #cpDB IDs for the two interaction partners
    id_a = row["multidata_1_id"]
    id_b = row["multidata_2_id"]
    
    # Convert each partner ID into the required gene set.
    partner_a = md2genes.get(id_a)
    partner_b = md2genes.get(id_b)

    classification = row["classification"]
    if pd.isna(classification):
        classification = ""

    records_by[cat].append((
        id_a,
        id_b,
        partner_a,
        partner_b,
        classification,
        row["id_cp_interaction"]))

# Print number of resolved interactions in each category
for cat, records in records_by.items():
    print(cat, len(records))

Adhesion-Adhesion 242
Ligand-Receptor 2508
Receptor-Receptor 28
Gap-Gap 10
Ligand-Ligand 123


In [19]:
summary = []

for cat, records in records_by.items():
    directed = cat == "Ligand-Receptor"
    binary_mat, count_mat = CCC.make_matrices(records, cluster_gene_sets, cluster_ids, directed)

    # Save both matrices with AGM names as row and column labels
    name = cat.replace("-", "_")
    pd.DataFrame(binary_mat, index=cluster_ids, columns=cluster_ids).to_csv(out / f"binary_{name}.csv")
    pd.DataFrame(count_mat, index=cluster_ids, columns=cluster_ids).to_csv(out / f"count_{name}.csv")

    # Undirected matrices store every A–B pair twice (as A–B and B–A),
    # so for those we only sum the upper triangle
    if directed:
        n_interactions = count_mat.sum()
        n_connections = binary_mat.sum()
    else:
        n_interactions = np.triu(count_mat).sum()
        n_connections = np.triu(binary_mat).sum()

    summary.append({
        "directionality": cat,
        "total_interactions": int(n_interactions),
        "total_cluster_connections": int(n_connections),
    })

summary_df = pd.DataFrame(summary)
summary_df.to_csv(out / "CCC_summary.csv", index=False)
summary_df

,directionality,total_interactions,total_cluster_connections
0,Adhesion-Adhesion,13,12
1,Ligand-Receptor,111,94
2,Receptor-Receptor,0,0
3,Gap-Gap,1,1
4,Ligand-Ligand,0,0


In [20]:
# Make one row per matched interaction for every AGM pair
def describe_partner(md_id, genes, side):
    """Describe one interaction partner; side is "sender" or "receiver"."""
    is_complex = bool(md2iscomplex.get(md_id, False))
    return {
        f"{side}_partner_multidata_id": md_id,
        f"{side}_partner_type": "complex" if is_complex else "single protein",
        f"{side}_partner_genes": ";".join(sorted(genes)),
        f"{side}_partner_component_proteins": complex_protein_components.get(md_id, "") if is_complex else "",
    }


out_rows = []

for cat, records in records_by.items():
    directed = cat == "Ligand-Receptor"

    for i, cluster_a in enumerate(cluster_ids):
        genes_a = cluster_gene_sets[cluster_a]

        # Directed: check every AGM pair. Undirected: each pair only once (A-B, not also B-A)
        for cluster_b in (cluster_ids if directed else cluster_ids[i:]):
            genes_b = cluster_gene_sets[cluster_b]

            for id_1, id_2, partner_1, partner_2, classification, interaction_id in records:
                forward = partner_1 <= genes_a and partner_2 <= genes_b
                reverse = partner_2 <= genes_a and partner_1 <= genes_b

                if forward:
                    sender_id, receiver_id = id_1, id_2
                    sender_genes, receiver_genes = partner_1, partner_2
                elif reverse and not directed:
                    # Found the other way round: flip so the sender partner is the one in cluster_a
                    sender_id, receiver_id = id_2, id_1
                    sender_genes, receiver_genes = partner_2, partner_1
                else:
                    continue  # no match for this AGM pair

                out_rows.append({
                    "directionality": cat,
                    "classification": classification,
                    "is_directed": directed,
                    "sender_cluster": cluster_a,
                    "receiver_cluster": cluster_b,
                    **describe_partner(sender_id, sender_genes, "sender"),
                    **describe_partner(receiver_id, receiver_genes, "receiver"),
                    # Original partner order in interaction_table.csv
                    "cellphonedb_multidata_1_id": id_1,
                    "cellphonedb_multidata_2_id": id_2,
                    "id_cp_interaction": interaction_id,
                })

lr_df = pd.DataFrame(out_rows)
lr_df.to_csv(out / "cluster_LR_pairs_all_classes.csv", index=False)
print(f"Saved {len(lr_df)} matched interaction rows")

Saved 125 matched interaction rows


In [21]:
# Confirm that the long-table row counts equal the matrix totals.
table_counts = (lr_df.groupby("directionality").size().rename("table_rows"))
validation = (summary_df.set_index("directionality")[["total_interactions"]].join(table_counts).fillna(0))
validation["matches"] = (validation["total_interactions"] == validation["table_rows"])
validation

,total_interactions,table_rows,matches
directionality,,,
Adhesion-Adhesion,13,13.0,True
Ligand-Receptor,111,111.0,True
Receptor-Receptor,0,0.0,True
Gap-Gap,1,1.0,True
Ligand-Ligand,0,0.0,True
